In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Spark Testing")
    .getOrCreate()
)

D:\Personal\Study\Learning\Data Engineer\De-learning-journey\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
customers_data = [
    (1, "Alice", "Bangalore", "India"),
    (2, "Bob", "Mumbai", "India"),
    (3, "Charlie", "Delhi", "India"),
    (4, "David", "Chennai", "India"),
    (5, "Emma", "Pune", "India"),
    (6, "Frank", "Hyderabad", "India"),
]

customers = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name", "city", "country"]
)

In [3]:
orders_data = [
    (101, 1, "2026-01-05", "Electronics", 50000, "Completed"),
    (102, 2, "2026-01-10", "Electronics", 30000, "Completed"),
    (103, 1, "2026-01-15", "Furniture", 15000, "Completed"),
    (104, 3, "2026-01-20", "Clothing", 5000, "Cancelled"),
    (105, 4, "2026-02-01", "Electronics", 45000, "Completed"),
    (106, 5, "2026-02-05", "Clothing", 7000, "Completed"),
    (107, 2, "2026-02-10", "Furniture", 20000, "Completed"),
    (108, 6, "2026-02-15", "Electronics", 60000, "Completed"),
    (109, 1, "2026-03-01", "Clothing", 8000, "Completed"),
    (110, 3, "2026-03-05", "Electronics", 40000, "Completed"),
    (111, 5, "2026-03-10", "Furniture", 25000, "Completed"),
    (112, 6, "2026-03-15", "Clothing", 9000, "Cancelled"),
]

orders = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "order_date",
     "category", "amount", "status"]
)

In [4]:
customers.show()
orders.show()

+-----------+-------------+---------+-------+
|customer_id|customer_name|     city|country|
+-----------+-------------+---------+-------+
|          1|        Alice|Bangalore|  India|
|          2|          Bob|   Mumbai|  India|
|          3|      Charlie|    Delhi|  India|
|          4|        David|  Chennai|  India|
|          5|         Emma|     Pune|  India|
|          6|        Frank|Hyderabad|  India|
+-----------+-------------+---------+-------+

+--------+-----------+----------+-----------+------+---------+
|order_id|customer_id|order_date|   category|amount|   status|
+--------+-----------+----------+-----------+------+---------+
|     101|          1|2026-01-05|Electronics| 50000|Completed|
|     102|          2|2026-01-10|Electronics| 30000|Completed|
|     103|          1|2026-01-15|  Furniture| 15000|Completed|
|     104|          3|2026-01-20|   Clothing|  5000|Cancelled|
|     105|          4|2026-02-01|Electronics| 45000|Completed|
|     106|          5|2026-02-05|  

In [5]:
# Q1 — Filter - Find all completed orders.
orders.filter(F.col("status") == "Completed").show()

+--------+-----------+----------+-----------+------+---------+
|order_id|customer_id|order_date|   category|amount|   status|
+--------+-----------+----------+-----------+------+---------+
|     101|          1|2026-01-05|Electronics| 50000|Completed|
|     102|          2|2026-01-10|Electronics| 30000|Completed|
|     103|          1|2026-01-15|  Furniture| 15000|Completed|
|     105|          4|2026-02-01|Electronics| 45000|Completed|
|     106|          5|2026-02-05|   Clothing|  7000|Completed|
|     107|          2|2026-02-10|  Furniture| 20000|Completed|
|     108|          6|2026-02-15|Electronics| 60000|Completed|
|     109|          1|2026-03-01|   Clothing|  8000|Completed|
|     110|          3|2026-03-05|Electronics| 40000|Completed|
|     111|          5|2026-03-10|  Furniture| 25000|Completed|
+--------+-----------+----------+-----------+------+---------+



In [6]:
# Q2 — Select columns order_id, customer_id, amount
orders.select("order_id", "customer_id", "amount").show()

+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|     101|          1| 50000|
|     102|          2| 30000|
|     103|          1| 15000|
|     104|          3|  5000|
|     105|          4| 45000|
|     106|          5|  7000|
|     107|          2| 20000|
|     108|          6| 60000|
|     109|          1|  8000|
|     110|          3| 40000|
|     111|          5| 25000|
|     112|          6|  9000|
+--------+-----------+------+



In [22]:
# Q3 — Amount filter Find orders where: amount > 30000 (Write SQL + PySpark.)
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
spark.sql("""
    SELECT *
    FROM orders
    WHERE amount > 30000
""").show()

+--------+-----------+----------+-----------+------+---------+
|order_id|customer_id|order_date|   category|amount|   status|
+--------+-----------+----------+-----------+------+---------+
|     101|          1|2026-01-05|Electronics| 50000|Completed|
|     105|          4|2026-02-01|Electronics| 45000|Completed|
|     108|          6|2026-02-15|Electronics| 60000|Completed|
|     110|          3|2026-03-05|Electronics| 40000|Completed|
+--------+-----------+----------+-----------+------+---------+



In [8]:
""" 
Q4 — Multiple conditions
Find completed orders where:

amount > 30000
AND category = 'Electronics'
"""
spark.sql(
    """
    select *
    from orders
    where amount > 30000
     and category = 'Electronics'
    """
).show()

# or

orders.filter(
    (F.col("amount") > 30000) &
    (F.col("category")  == "Electronics")
).show()


+--------+-----------+----------+-----------+------+---------+
|order_id|customer_id|order_date|   category|amount|   status|
+--------+-----------+----------+-----------+------+---------+
|     101|          1|2026-01-05|Electronics| 50000|Completed|
|     105|          4|2026-02-01|Electronics| 45000|Completed|
|     108|          6|2026-02-15|Electronics| 60000|Completed|
|     110|          3|2026-03-05|Electronics| 40000|Completed|
+--------+-----------+----------+-----------+------+---------+

+--------+-----------+----------+-----------+------+---------+
|order_id|customer_id|order_date|   category|amount|   status|
+--------+-----------+----------+-----------+------+---------+
|     101|          1|2026-01-05|Electronics| 50000|Completed|
|     105|          4|2026-02-01|Electronics| 45000|Completed|
|     108|          6|2026-02-15|Electronics| 60000|Completed|
|     110|          3|2026-03-05|Electronics| 40000|Completed|
+--------+-----------+----------+-----------+------+--

In [9]:
"""
Q5 — Sort all orders by amount in descending order.
Give me: SQL, PySpark
"""
# SQL
"""
select *
from orders
order by amount desc
"""
# spark
orders.orderBy(
    F.col("amount").desc()
).show()



+--------+-----------+----------+-----------+------+---------+
|order_id|customer_id|order_date|   category|amount|   status|
+--------+-----------+----------+-----------+------+---------+
|     108|          6|2026-02-15|Electronics| 60000|Completed|
|     101|          1|2026-01-05|Electronics| 50000|Completed|
|     105|          4|2026-02-01|Electronics| 45000|Completed|
|     110|          3|2026-03-05|Electronics| 40000|Completed|
|     102|          2|2026-01-10|Electronics| 30000|Completed|
|     111|          5|2026-03-10|  Furniture| 25000|Completed|
|     107|          2|2026-02-10|  Furniture| 20000|Completed|
|     103|          1|2026-01-15|  Furniture| 15000|Completed|
|     112|          6|2026-03-15|   Clothing|  9000|Cancelled|
|     109|          1|2026-03-01|   Clothing|  8000|Completed|
|     106|          5|2026-02-05|   Clothing|  7000|Completed|
|     104|          3|2026-01-20|   Clothing|  5000|Cancelled|
+--------+-----------+----------+-----------+------+---

In [10]:
"""
Q6 — Find the total sales amount across all orders.
Expected SQL concept: SUM(amount)
"""
# SQL
"""
select
sum(amount) as total_sales_amount
from orders
"""

# spark

orders.agg(
    F.sum("amount").alias("total_sales_amount")
).show()

+------------------+
|total_sales_amount|
+------------------+
|            314000|
+------------------+



In [11]:
"""
Q7 — Find the average order amount across all orders.
"""

# sql
"""
select
avg(amount) as avg_amount
from orders
"""

# spark
orders.select(
    F.avg("amount").alias("avg_amount")
).show()

+------------------+
|        avg_amount|
+------------------+
|26166.666666666668|
+------------------+



In [12]:
"""
Q8 — Find total sales for each category.

For example, conceptually:

Electronics → total
Furniture   → total
Clothing    → total
"""

# sql
"""
select
    category, sum(amount) as total
from orders
group by category;
"""

# spark
orders.groupBy("category").agg(
    F.sum("amount").alias("total")
).show()


+-----------+------+
|   category| total|
+-----------+------+
|Electronics|225000|
|  Furniture| 60000|
|   Clothing| 29000|
+-----------+------+



In [13]:
"""
Q9 — Find the number of orders for each category.

Write both SQL and PySpark.

Hint: you'll need COUNT() in SQL and F.count() in PySpark.
"""
# sql
"""
SELECT
    category,
    COUNT(*) AS order_count
FROM orders
GROUP BY category;
"""
orders.groupBy("category").agg(
    F.count("category").alias("order_count")
).show()

+-----------+-----------+
|   category|order_count|
+-----------+-----------+
|Electronics|          5|
|  Furniture|          3|
|   Clothing|          4|
+-----------+-----------+



In [14]:
"""
Q10 — Find the total sales for each category, but only for Completed orders.
"""
# sql
"""
select category, sum(amount) as total_sales
from orders
group by category
where status = 'Completed'
"""
orders.filter(
    (F.col("status") == "Completed")
).groupBy("category").agg(
    F.sum("amount").alias("total_sales")
).show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     225000|
|  Furniture|      60000|
|   Clothing|      15000|
+-----------+-----------+



In [15]:
"""
Q11 — Find categories where total completed sales are greater than ₹50,000.
"""
# sql
"""
select category, sum(amount) as total_sales
from orders
where status = 'Completed'
group by category
having total_sales > 50000
"""

orders.filter(
    (F.col("status") == "Completed")
).groupBy("category").agg(
    F.sum("amount").alias("total_sales")
).filter(
    F.col("total_sales") > 50000
).show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     225000|
|  Furniture|      60000|
+-----------+-----------+



In [19]:
"""
Q12 — Find customers who have placed more than 1 completed order.
"""
# sql
"""
select customer_id, count(*) as total_counts
from orders
where status = 'Completed'
group by customer_id
having count(*) > 1;
"""

orders.filter(
    F.col("status") == "Completed"
).groupBy("customer_id").agg(
    F.count("order_id").alias("total_count")
).filter(
    F.col("total_count") > 1
).show()

+-----------+-----------+
|customer_id|total_count|
+-----------+-----------+
|          1|          3|
|          2|          2|
|          5|          2|
+-----------+-----------+



In [25]:
"""
Q13 — JOIN

Using: orders, customers

Find:

order_id
customer_name
city
amount

"""
# sql
"""
select 
    o.order_id,
    c.customer_name,
    c.city,
    o.amount
from orders o
join customers c
    on c.customer_id = o.customer_id
"""

orders.join(
    customers, 
    orders.customer_id == customers.customer_id, "inner"
).select(
    "order_id", "customer_name", "city", "amount"
).show()


orders.join(
    customers, 
    on="customer_id", how="inner"
).select(
    "order_id", "customer_name", "city", "amount"
).show()

+--------+-------------+---------+------+
|order_id|customer_name|     city|amount|
+--------+-------------+---------+------+
|     101|        Alice|Bangalore| 50000|
|     103|        Alice|Bangalore| 15000|
|     109|        Alice|Bangalore|  8000|
|     102|          Bob|   Mumbai| 30000|
|     107|          Bob|   Mumbai| 20000|
|     104|      Charlie|    Delhi|  5000|
|     110|      Charlie|    Delhi| 40000|
|     105|        David|  Chennai| 45000|
|     106|         Emma|     Pune|  7000|
|     111|         Emma|     Pune| 25000|
|     108|        Frank|Hyderabad| 60000|
|     112|        Frank|Hyderabad|  9000|
+--------+-------------+---------+------+

+--------+-------------+---------+------+
|order_id|customer_name|     city|amount|
+--------+-------------+---------+------+
|     101|        Alice|Bangalore| 50000|
|     103|        Alice|Bangalore| 15000|
|     109|        Alice|Bangalore|  8000|
|     102|          Bob|   Mumbai| 30000|
|     107|          Bob|   Mumbai

In [37]:
"""
Q14 — Find the total completed sales for each customer.

"""
# sql
"""
select customer_id, sum(amount) as total_sales
from orders
where status = 'Completed'
group by customer_id
"""

orders.filter(
    F.col("status") == "Completed"
).groupBy("customer_id").agg(
    F.sum("amount").alias("total_sales")
).show()


+-----------+-----------+
|customer_id|total_sales|
+-----------+-----------+
|          1|      73000|
|          2|      50000|
|          5|      32000|
|          4|      45000|
|          6|      60000|
|          3|      40000|
+-----------+-----------+



In [38]:
"""
Q15 — Find customers whose total completed sales are greater than 30,000.

Expected columns:

customer_id
customer_name
total_sales

"""
# sql
"""
select
    c.customer_id,
    c.customer_name,
    sum("amount") as total_sales
from orders o
join customers c
    on c.customer_id = o.customer_id
WHERE o.status = 'Completed'
group by c.customer_id, c.customer_name
having total_sales > 30000
"""

orders.filter(
    F.col("status") == "Completed"
).join(
    customers,
    on="customer_id",
    how="inner"
).groupBy(
    "customer_id", "customer_name"
).agg(
    F.sum("amount").alias("total_sales")
).filter(F.col("total_sales") > 30000).show()
    

+-----------+-------------+-----------+
|customer_id|customer_name|total_sales|
+-----------+-------------+-----------+
|          1|        Alice|      73000|
|          2|          Bob|      50000|
|          3|      Charlie|      40000|
|          4|        David|      45000|
|          5|         Emma|      32000|
|          6|        Frank|      60000|
+-----------+-------------+-----------+

